# GPU support in scikit-learn with the Array API

## Example of a complex non-linear regression pipeline

At the time of writing, this notebook requires experimental features only available in the developer version of scikit-learn:

In [ ]:
# %pip uninstall -q -y sklearn-compat
# %pip install -q -U --pre --extra-index https://pypi.anaconda.org/scientific-python-nightly-wheels/simple scikit-learn

Let's enable the Array API support in SciPy and scikit-learn. See the scikit-learn documentation for more details:

https://scikit-learn.org/dev/modules/array_api.html

In [ ]:
import os
import sklearn

os.environ["SCIPY_ARRAY_API"] = "1"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
sklearn.set_config(array_api_dispatch=True)

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)  # matmul divide by zero on macOS...
warnings.filterwarnings("ignore", category=UserWarning)  # n_components > n_samples

Let's define a low dimensional (2D) regression problem. Both the mean and variance of the target variable depend on the input features.

In [ ]:
import numpy as np
import torch


def true_mean(X):
    return np.sin(X[:, 0] * 2) * np.cos(X[:, 1]) ** 4


def true_std(X):
    return 0.3 * np.cos(X[:, 1]) ** 6 + 0.1


def make_data(n_samples=int(1e5), seed=0):
    rng = np.random.default_rng(seed=seed)
    X = rng.uniform(low=-3, high=3, size=(n_samples, 2))
    y = rng.normal(loc=true_mean(X), scale=true_std(X))
    return X.astype(np.float32), y.astype(np.float32)


X, y = make_data()


In [ ]:
if torch.backends.cuda.is_built():
    gpu_device = "cuda"
elif torch.backends.mps.is_built():
    gpu_device = "mps"
else:
    print("No GPU device found, falling back to CPU...")
    gpu_device = "cpu"

X_torch_gpu = torch.asarray(X, device=gpu_device)
y_torch_gpu = torch.asarray(y, device=gpu_device)
gpu_device


The **non-linear interactions** of the features makes the regression problem more complex: a simple linear regression model would not be well-specified for this task.

The feature dependent variance of the target variable makes this problem an **heteroscedastic** regression problem. This would be important to keep in mind if we were to model prediction uncertainty but we will leave that aside for this notebook and only focus on modeling the conditional mean of the target.

In [ ]:
import matplotlib.pyplot as plt

x1 = np.linspace(-3, 3, 200).astype(np.float32)
x2 = np.linspace(-3, 3, 200).astype(np.float32)
X1, X2 = np.meshgrid(x1, x2)

fig, axs = plt.subplots(ncols=3, figsize=(15, 5))
axs[0].pcolormesh(X1, X2, true_mean(np.c_[X1.ravel(), X2.ravel()]).reshape(*X1.shape))
axs[0].set(title="True mean E[Y|X]", xlabel="X1", ylabel="X2")
axs[1].pcolormesh(X1, X2, true_std(np.c_[X1.ravel(), X2.ravel()]).reshape(*X1.shape) ** 2)
axs[1].set(title="True variance Var[Y|X]", xlabel="X1", ylabel="X2")
axs[2].scatter(X[:, 0], X[:, 1], c=y, s=5)
axs[2].set(title="Observations $(y_i, X_i)$", xlabel="X1", ylabel="X2");

This is a medium-size machine learning problem. The total dataset contains 100k observations in a 2D space. This represents less than a MB of data.

In [ ]:
X.shape

In [ ]:
X.nbytes / 1e6

Let's estimate the Bayes error rate for this data generating process. The vast majority of the variability of the target `y` can be explained by the input features in `X`.

In [ ]:
from sklearn.metrics import r2_score

r2_score(y, true_mean(X))

In the following we will attempt to compose scikit-learn building block to train a model that estimates this mean function as accurately as possible from the available data points.

We first consider a polynomial regression pipeline fit on the CPU using the default NumPy & SciPy-based implementations that comes with scikit-learn:

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import RidgeCV
from sklearn.kernel_approximation import Nystroem
from sklearn.model_selection import cross_validate


poly_reg_numpy_cpu = make_pipeline(
    SplineTransformer(n_knots=5),
    Nystroem(kernel="poly", degree=2, n_components=300, random_state=0),
    RidgeCV(alphas=np.logspace(-6, 6, 13)),
)
poly_reg_numpy_cpu

In [ ]:
%%time
cv_results_numpy_cpu = cross_validate(poly_reg_numpy_cpu, X, y, cv=5)
pd.DataFrame(cv_results_numpy_cpu)[["test_score", "fit_time", "score_time"]].round(3)

Fitting and evaluating the pipeline 5 times on the 2 CPU cores takes around 30 s on the base Google Colab host and 10 s with more powerful runtimes.

Note that this pipeline has many hyper-parameters to adjust. Tweaking them interactively with a 30 s long iterations is not feasible: the data scientist will often loose their focus.

Conducting a systematic hyper-parameter tuning sweap would require more CPUs on a beafier dedicated machine and would also break the interactive user experience.

Let's instead adapt the pipeline to move the last two steps (which happen to be the linear-algebra heavy steps) to the GPU with the help of PyTorch:

In [ ]:
%%time
from sklearn.preprocessing import FunctionTransformer
from functools import partial


poly_reg_torch_gpu = make_pipeline(
    SplineTransformer(n_knots=5),
    FunctionTransformer(partial(torch.asarray, device=gpu_device)),
    Nystroem(kernel="poly", degree=2, n_components=300, random_state=0),
    RidgeCV(alphas=np.logspace(-6, 6, 13)),
)

cv_results_torch_gpu = cross_validate(
    poly_reg_torch_gpu, X, y_torch_gpu, cv=5, return_estimator=True, error_score="raise"
)
pd.DataFrame(cv_results_torch_gpu)[["test_score", "fit_time", "score_time"]].round(3)

Let's qualitatively check the learned prediction function of the models trained for the each iteration of this cross-validation:

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(9, 6), constrained_layout=True)
axs[0, 0].pcolormesh(X1, X2, true_mean(np.c_[X1.ravel(), X2.ravel()]).reshape(*X1.shape))
axs[0, 0].set(title="True mean", xlabel="X1", ylabel="X2")
for est, ax in zip(cv_results_torch_gpu["estimator"], axs.ravel()[1:]):
    ax.pcolormesh(X1, X2, est.predict(np.c_[X1.ravel(), X2.ravel()]).reshape(*X1.shape).cpu())
    ax.set(title="Model predictions", xlabel="X1", ylabel="X2");

The resulting pipeline is both qualitatitively and quantitative far from optimal.

But since fitting this pipeline is now much faster, let's try to tune the hyperparameters:

In [ ]:
%%time
from sklearn.model_selection import RandomizedSearchCV


param_grid = dict(
    splinetransformer__n_knots=range(3, 30),
    nystroem__kernel=["poly", "rbf"],
    nystroem__degree=[2, 3, 4, 5],
    nystroem__gamma=np.logspace(-6, 6, 100),
    nystroem__n_components=[50, 100, 200, 500],
    # ridge__alpha=np.logspace(-6, 6, 100),
)

search_cv = RandomizedSearchCV(
    poly_reg_torch_gpu,
    param_grid,
    n_iter=30,
    random_state=0,
)
search_cv.fit(X, y_torch_gpu)

In [ ]:
search_cv.best_score_.round(3)

The cross-validation score of the best model is very close to the Bayes optimal score computed from our knowledge of the data-generating process!

Let's qualitatively compare our mean estimate to the ground truth of the data generating process.

In [ ]:
best_model = search_cv.best_estimator_

fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
axs[0].pcolormesh(X1, X2, true_mean(np.c_[X1.ravel(), X2.ravel()]).reshape(*X1.shape))
axs[0].set(title="True mean", xlabel="X1", ylabel="X2")
axs[1].pcolormesh(X1, X2, best_model.predict(np.c_[X1.ravel(), X2.ravel()]).reshape(*X1.shape).cpu())
axs[1].set(title="Tuned pipeline predictions", xlabel="X1", ylabel="X2");

We see a very good match: it's hard to visually identify any defect once the hyparameters have been tuned.

Thanks to **GPU** acceleration, we can **tune the hyperparamters** a complex pipeline to get a very good model **in the time it would take to run a single cross-validation** on the Google Colab CPU.

More importantly, the training speed is fast enough to **avoid disrupting the model development flow** of the data scientist editing the Google Colab notebook **interactively**.


Let's have a look at the optimal hyper-parameter combinations:

In [ ]:
search_cv.best_params_

In [ ]:
search_cv.best_estimator_[-1].alpha_

For completeness here are the other hyperparameters of the leading models in the search:

In [ ]:
pd.DataFrame(
    search_cv.cv_results_
).sort_values("mean_test_score", ascending=False).head().round(3)

## Out of curiosity, let's compare with TabICLv2

In [ ]:
%pip install -U -q tabicl

In [ ]:
from tabicl.sklearn.regressor import TabICLRegressor
from time import perf_counter

from sklearn.model_selection import LearningCurveDisplay, ShuffleSplit
import tabicl

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 6), sharey=True)

common_params = {
    "X": X,
    "y": y,
    "cv": ShuffleSplit(n_splits=3, test_size=0.2, random_state=0),
    "score_type": "both",
    "line_kw": {"marker": "o"},
    "std_display_style": "fill_between",
    "score_name": "R2",
}
tuned_poly_reg_cpu = make_pipeline(
    SplineTransformer(n_knots=10),
    Nystroem(kernel="poly", degree=2, n_components=300, random_state=0),
    RidgeCV(alphas=np.logspace(-6, 6, 13)),
)

for ax_idx, (estimator, train_sizes) in enumerate(
    [
        (tuned_poly_reg_cpu, [100, 200, 400, 800, 1600, 3200]),
        (tabicl.TabICLRegressor(), [100, 200, 400]),
    ]
):
    print("Computing learning curve for", estimator.__class__.__name__)
    tic = perf_counter()
    LearningCurveDisplay.from_estimator(
        estimator, **common_params, ax=ax[ax_idx], error_score="raise"
    )
    toc = perf_counter()
    print(f"Time taken: {toc - tic:.2f} seconds")
    handles, label = ax[ax_idx].get_legend_handles_labels()
    ax[ax_idx].legend(handles[:2], ["Training Score", "Test Score"])
    ax[ax_idx].set_title(f"Learning Curve for {estimator.__class__.__name__}")


In [ ]:
X_subset = X[:1000]
y_subset = y[:1000]

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from scipy.stats import norm
from sklearn.linear_model import TweedieRegressor
from sklearn.model_selection import GridSearchCV


class GaussianQuantileRegressorCV(BaseEstimator, RegressorMixin):
    """Assume that the residuals are Gaussian distributed"""

    def __init__(
        self,
        alphas=np.logspace(-6, 6, 13),
    ):
        self.alphas = alphas

    def fit(self, X, y, sample_weight=None):
        # TODO: expose mean estimator as a parameter and implement
        # cross-fitting procedure to compute the squared residuals if the mean
        # estimator is not a `RidgeCV` with the right choice of `cv` and
        # `scoring` params.
        mean_estimator = RidgeCV(alphas=self.alphas, store_cv_results=True)
        best_alpha_idx = np.flatnonzero(np.asarray(self.alphas) == self.alphas)[0]

        # TODO: expose var estimator as a parameter.
        # Note: TweedieRegressor power=2 should be a good choice for the variance
        # estimator: if the residuals are Gaussian distributed, their squared
        # values are chi-squared distributed and therefore a particular case of
        # the Gamma (Tweedie power=2) distribution. However, in practice power=1.5
        # seems to work a bit better.
        var_estimator = TweedieRegressor(alpha=1e-12, max_iter=1_000, power=1.5)
        # var_estimator = GridSearchCV(
        #     TweedieRegressor(alpha=1e-6, power=2, max_iter=1_000),
        #     param_grid={"alpha": np.logspace(-12, 12, 13), "power": [1.1, 1.3, 1.5, 1.7, 2]},
        #     n_jobs=4,
        # )

        self.mean_estimator_ = mean_estimator.fit(X, y, sample_weight=sample_weight)
        squared_residuals = self.mean_estimator_.cv_results_[:, best_alpha_idx]
        self.var_estimator_ = var_estimator.fit(
            X, squared_residuals, sample_weight=sample_weight
        )
        return self

    def predict(self, X, output_type="mean", alphas=None):
        """Compute conditional distribution parameters.

        If output_type == "mean" or "median" return an array of shape (n_samples,).

        If output_type == "quantile" return an array of shape (n_samples, n_quantiles).

        Since the model assumes Gaussian Y|X, mean and median are the same.

        The variance is computed as exp(log_var), which is the conditional
        variance of Y|X. As a result, any quantile of Y|X is a linear
        transformation of the quantiles of the residuals.
        """
        if output_type in ["mean", "median"]:
            return self.mean_estimator_.predict(X)
        elif output_type == "quantile":
            if alphas is None:
                alphas = [0.1, 0.25, 0.5, 0.75, 0.9]
            alphas = np.asarray(alphas)
            n_samples = X.shape[0]

            mean_X = self.mean_estimator_.predict(X).reshape(n_samples, 1)
            var_X = self.var_estimator_.predict(X).reshape(n_samples, 1)
            return mean_X + np.sqrt(var_X) * norm.ppf(alphas).reshape(
                1, alphas.shape[0]
            )


poly_dist_reg = make_pipeline(
    SplineTransformer(n_knots=20),
    Nystroem(kernel="poly", degree=2, n_components=500, random_state=0),
    GaussianQuantileRegressorCV(),
)
poly_dist_reg.fit(X, y)
poly_dist_reg.predict(
    X, output_type="quantile", alphas=[0.1, 0.25, 0.5, 0.75, 0.9]
).shape

In [ ]:
# poly_dist_reg[-1].var_estimator_.best_params_

In [ ]:
# import pandas as pd

# pd.DataFrame(poly_dist_reg[-1].var_estimator_.cv_results_).sort_values("mean_test_score", ascending=False)

In [ ]:
X_test, y_test = make_data(n_samples=int(3e4), seed=2)

predicted_quartiles = poly_dist_reg.predict(X_test, output_type="quantile", alphas=[0.25, 0.75])
predicted_iqr = predicted_quartiles[:, 1] - predicted_quartiles[:, 0]

true_quartiles = true_std(X_test).reshape(-1, 1) * norm.ppf([0.25, 0.75]).reshape(1, 2)
true_iqr = true_quartiles[:, 1] - true_quartiles[:, 0]

bins = np.linspace(0, 0.3, 30)

plt.hist(predicted_iqr, bins=bins, alpha=0.5, label="Predicted")
plt.hist(true_iqr, bins=bins, alpha=0.5, label="True")
plt.title("Histogram of the inter-quantile range of Y|X on test data")
_ = plt.legend()

Plot the IQR map of the poly reg model over the feature space and along side the ground-truth derived from the data generating process:

In [ ]:
x1 = np.linspace(-3, 3, 200).astype(np.float32)
x2 = np.linspace(-3, 3, 200).astype(np.float32)
X1, X2 = np.meshgrid(x1, x2)

fig, axs = plt.subplots(ncols=2, figsize=(15, 5), constrained_layout=True)
true_quartiles = true_std(np.c_[X1.ravel(), X2.ravel()]).reshape(-1, 1) * norm.ppf([0.25, 0.75]).reshape(1, 2)
true_iqr = true_quartiles[:, 1] - true_quartiles[:, 0]
mesh_true_iqr = axs[0].pcolormesh(X1, X2, true_iqr.reshape(*X1.shape))
axs[0].set(title="True IQR", xlabel="X1", ylabel="X2")

poly_reg_quartiles = poly_dist_reg.predict(np.c_[X1.ravel(), X2.ravel()], output_type="quantile", alphas=[0.25, 0.75])
poly_reg_iqr = poly_reg_quartiles[:, 1] - poly_reg_quartiles[:, 0]
axs[1].pcolormesh(X1, X2, poly_reg_iqr.reshape(*X1.shape))
axs[1].set(title="Poly dist reg IQR", xlabel="X1", ylabel="X2")

_ = fig.colorbar(mesh_true_iqr, ax=axs, shrink=0.8, aspect=30, label='IQR')

In [ ]:
_ = plt.hist(true_iqr, bins=np.linspace(0, 0.3, 30), alpha=0.5, label="True IQR")
_ = plt.hist(poly_reg_iqr, bins=np.linspace(0, 0.3, 30), alpha=0.5, label="Poly dist reg IQR")
_ = plt.legend()

In [ ]:
pd.DataFrame(cross_validate(tabicl.TabICLRegressor(n_estimators=1), X_subset, y_subset, cv=5))


In [ ]:
pd.DataFrame(cross_validate(tabicl.TabICLRegressor(n_estimators=2), X_subset, y_subset, cv=5))


In [ ]:
pd.DataFrame(cross_validate(tabicl.TabICLRegressor(n_estimators=8), X_subset, y_subset, cv=5))


In [ ]:
treg = tabicl.TabICLRegressor(n_estimators=8).fit(X_subset, y_subset)
predicted_quartiles_treg = treg.predict(X_test, output_type="quantiles", alphas=[0.25, 0.75])
predicted_quartiles_treg

In [ ]:
plt.hist(predicted_quartiles_treg[:, -1] - predicted_quartiles_treg[:, 0], bins=bins, alpha=0.5, label="TabICL")
plt.hist(true_iqr, bins=bins, alpha=0.5, label="True")
plt.title("Histogram of the inter-quantile range of Y|X on test data")
_ = plt.legend()


In [ ]:
x1 = np.linspace(-3, 3, 200).astype(np.float32)
x2 = np.linspace(-3, 3, 200).astype(np.float32)
X1, X2 = np.meshgrid(x1, x2)

fig, axs = plt.subplots(ncols=2, figsize=(15, 5), constrained_layout=True)
true_quartiles = true_std(np.c_[X1.ravel(), X2.ravel()]).reshape(-1, 1) * norm.ppf([0.25, 0.75]).reshape(1, 2)
true_iqr = true_quartiles[:, 1] - true_quartiles[:, 0]
mesh_true_iqr = axs[0].pcolormesh(X1, X2, true_iqr.reshape(*X1.shape))
axs[0].set(title="True IQR", xlabel="X1", ylabel="X2")

treg_quartiles = treg.predict(np.c_[X1.ravel(), X2.ravel()], output_type="quantiles", alphas=[0.25, 0.75])
treg_iqr = treg_quartiles[:, 1] - treg_quartiles[:, 0]
axs[1].pcolormesh(X1, X2, treg_iqr.reshape(*X1.shape))
axs[1].set(title="TabICL IQR", xlabel="X1", ylabel="X2")

_ = fig.colorbar(mesh_true_iqr, ax=axs, shrink=0.8, aspect=30, label='IQR')